# 05 SHAP可解释性分析
**论文章节对应**：第5章 模型解释与特征重要性分析

本notebook完成以下任务：
1. 对最优模型计算SHAP值
   - 若最优模型为XGBoost/RF：直接使用TreeExplainer
   - 若最优模型为Stacking：对两个基学习器分别计算SHAP，按Ridge系数加权合并
2. 绘制SHAP蜂群图（图5-4）：全局特征重要性
3. 绘制SHAP依赖图（图5-5）：关键特征边际效应
4. 绘制SHAP瀑布图（图5-6）：单个项目预测拆解

**输入**：`outputs/saved_models/` 中的模型文件
**输出**：图5-4/5-5/5-6

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import build_preprocessor, get_feature_names

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
DPI = 300

# ── 读取数据与模型信息 ────────────────────────────
train_df    = pd.read_csv('data/processed/train.csv', encoding='utf-8-sig')
test_df     = pd.read_csv('data/processed/test.csv',  encoding='utf-8-sig')
config      = joblib.load('outputs/saved_models/feature_config.pkl')
model_info  = joblib.load('outputs/saved_models/best_model_info.pkl')
preprocessor = joblib.load('outputs/saved_models/preprocessor.pkl')

all_features     = config['all_features']
best_model_name  = model_info['best_model_name']

X_train = train_df[all_features]
X_test  = test_df[all_features]
y_test  = model_info['y_test']

# 预处理（已经在notebook 03 fit过，这里直接transform）
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# 获取处理后的特征名
try:
    feature_names = preprocessor.get_feature_names_out()
    # 简化特征名（去掉前缀）
    feature_names = [n.split('__')[-1] for n in feature_names]
except Exception:
    feature_names = [f'feature_{i}' for i in range(X_test_proc.shape[1])]

print(f'最优模型: {best_model_name}')
print(f'测试集样本数: {len(X_test_proc)}')
print(f'处理后特征数: {len(feature_names)}')

## 5.1 加载最优模型并提取裸模型（用于SHAP）

In [ ]:
# 根据最优模型类型选择SHAP计算策略
def get_shap_values_for_best_model(best_model_name, X_proc, feature_names):
    """
    根据最优模型类型计算SHAP值
    - XGBoost/RF：TreeExplainer直接计算
    - Blending：无法直接用TreeExplainer，使用各模型加权
    - Stacking：对RF和XGB分别计算SHAP，按Ridge系数加权合并
    """
    if 'XGBoost' in best_model_name or 'Blending' in best_model_name:
        # 从Pipeline中提取XGB裸模型
        best_xgb_pipeline = joblib.load('outputs/saved_models/xgb_best.pkl')
        xgb_model = best_xgb_pipeline.named_steps['regressor']
        
        print('使用XGBoost TreeExplainer计算SHAP...')
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_proc)
        base_value  = explainer.expected_value
        model_label = 'XGBoost'
        
    elif '随机森林' in best_model_name:
        # 从Pipeline中提取RF裸模型
        best_rf_pipeline = joblib.load('outputs/saved_models/rf_best.pkl')
        rf_model = best_rf_pipeline.named_steps['regressor']
        
        print('使用随机森林 TreeExplainer计算SHAP...')
        explainer = shap.TreeExplainer(rf_model)
        shap_values = explainer.shap_values(X_proc)
        base_value  = explainer.expected_value
        model_label = '随机森林'
        
    elif 'Stacking' in best_model_name:
        print('Stacking模型：分别计算RF和XGB的SHAP，按Ridge系数加权合并...')
        stacking = joblib.load('outputs/saved_models/stacking_best.pkl')
        
        rf_model  = stacking.estimators_[0]
        xgb_model = stacking.estimators_[1]
        ridge_coefs = stacking.final_estimator_.coef_
        
        print(f'  Ridge元学习器系数: RF={ridge_coefs[0]:.3f}, XGB={ridge_coefs[1]:.3f}')
        
        # 分别计算SHAP
        shap_rf  = shap.TreeExplainer(rf_model).shap_values(X_proc)
        shap_xgb = shap.TreeExplainer(xgb_model).shap_values(X_proc)
        
        # 按Ridge系数加权合并
        shap_values = ridge_coefs[0] * shap_rf + ridge_coefs[1] * shap_xgb
        base_value  = (ridge_coefs[0] * shap.TreeExplainer(rf_model).expected_value +
                       ridge_coefs[1] * shap.TreeExplainer(xgb_model).expected_value)
        model_label = 'Stacking（加权SHAP）'
    else:
        raise ValueError(f'未知模型类型: {best_model_name}')
    
    return shap_values, base_value, model_label


shap_values, base_value, model_label = get_shap_values_for_best_model(
    best_model_name, X_test_proc, feature_names
)
print(f'✓ SHAP值计算完成，形状: {shap_values.shape}')

## 5.2 图5-4：SHAP蜂群图（全局特征重要性）

In [ ]:
print('绘制图5-4：SHAP蜂群图...')

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=feature_names,
    plot_type='dot',    # beeswarm蜂群图
    max_display=15,     # 展示前15个重要特征
    show=False,
)
plt.title(f'图5-4 SHAP蜂群图 — {model_label}', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/figures/fig5_4_shap_beeswarm.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('✓ 图5-4已保存')

In [ ]:
# 打印SHAP特征重要性排序（全局均值|SHAP|）
mean_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    '特征': feature_names,
    '平均|SHAP|': mean_shap,
}).sort_values('平均|SHAP|', ascending=False).reset_index(drop=True)

print('\nSHAP特征重要性排序（Top 10）：')
print(importance_df.head(10).to_string(index=False))

# 选出最重要的2-3个特征用于依赖图
top_features = importance_df.head(3)['特征'].tolist()
top_indices  = [list(feature_names).index(f) for f in top_features if f in feature_names]
print(f'\n将用以下特征绘制依赖图: {top_features[:3]}')

## 5.3 图5-5：SHAP依赖图（关键特征边际效应）

In [ ]:
print('绘制图5-5：SHAP依赖图（前3个重要特征）...')

n_dep = min(3, len(top_indices))
fig, axes = plt.subplots(1, n_dep, figsize=(6 * n_dep, 5))

if n_dep == 1:
    axes = [axes]

for ax, idx in zip(axes, top_indices[:n_dep]):
    feat_name = feature_names[idx]
    
    # 自动选择颜色交互特征（SHAP dependence plot标准做法）
    shap.dependence_plot(
        idx,
        shap_values,
        X_test_proc,
        feature_names=feature_names,
        ax=ax,
        show=False,
    )
    ax.set_title(f'依赖图: {feat_name}', fontsize=11)

plt.suptitle('图5-5 SHAP依赖图（关键特征边际效应）', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/fig5_5_shap_dependence.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('✓ 图5-5已保存')

## 5.4 图5-6：SHAP瀑布图（单个典型项目预测拆解）

In [ ]:
print('绘制图5-6：SHAP瀑布图（选取典型项目）...')

# 选取预测误差最小的样本作为"典型项目"展示
best_pred_all = model_info['best_pred']
errors = np.abs(best_pred_all - y_test)
typical_idx = np.argmin(errors)  # 预测最准的样本

print(f'选取第{typical_idx}号样本（预测误差最小）')
print(f'  实际造价: {np.expm1(y_test[typical_idx]):.0f} 元/㎡')
print(f'  预测造价: {np.expm1(best_pred_all[typical_idx]):.0f} 元/㎡')

# 构建SHAP Explanation对象
explanation = shap.Explanation(
    values=shap_values[typical_idx],
    base_values=base_value,
    data=X_test_proc[typical_idx],
    feature_names=feature_names,
)

# 绘制瀑布图
plt.figure(figsize=(10, 7))
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title(f'图5-6 SHAP瀑布图 — 典型项目预测拆解\n（样本#{typical_idx}）', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/fig5_6_shap_waterfall.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('✓ 图5-6已保存')

## 5.5 SHAP特征重要性补充分析

In [ ]:
# Bar plot（平均SHAP绝对值，作为图5-4的补充）
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=feature_names,
    plot_type='bar',
    max_display=12,
    show=False,
)
plt.title('SHAP特征重要性（平均|SHAP|）', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/figures/fig5_4b_shap_bar.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('✓ SHAP Bar图已保存')

In [ ]:
# 保存SHAP结果摘要
importance_df.to_csv('outputs/shap_importance.csv', index=False, encoding='utf-8-sig')

print('\n=== Notebook 05 完成 ===')
print('生成图表: 图5-4（蜂群图）/ 图5-5（依赖图）/ 图5-6（瀑布图）')
print(f'\nSHAP分析最重要的3个特征:')
for i, row in importance_df.head(3).iterrows():
    print(f'  {i+1}. {row["特征"]}: 平均|SHAP| = {row["平均|SHAP|"]:.4f}')

print('\n=== 全部论文图表已生成 ===')
print('3章图表: 图3-1（相关性热力图）/ 图3-2（VIF检验）')
print('4章图表: 图4-1（分布对比）/ 图4-2（省份箱线图）/ 图4-3（数据概况）')
print('5章图表: 图5-1（模型对比）/ 图5-2（预测vs实际）/ 图5-3（残差图）')
print('         图5-4（SHAP蜂群）/ 图5-5（SHAP依赖）/ 图5-6（SHAP瀑布）/ 图5-7（学习曲线）')